In [1]:
import sys
import ollama
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QTextEdit,
    QPushButton, QHBoxLayout, QSizePolicy
)
from PyQt5.QtCore import QThread, pyqtSignal
from PyQt5.QtGui import QFont


class AIWorker(QThread):
    token_received = pyqtSignal(str)
    finished = pyqtSignal()

    def __init__(self, messages, remember):
        super().__init__()
        self.messages = messages
        self.remember = remember

    def run(self):
        if self.remember:
            response = ollama.chat(
                model="deepseek-r1:8b",
                messages=self.messages,
                stream=True
            )
        else:
            response = ollama.chat(
                model="deepseek-r1:8b",
                messages=[self.messages[-1]],
                stream=True
            )

        for chunk in response:
            token = chunk["message"]["content"]
            self.token_received.emit(token)

        self.finished.emit()


class ChatApp(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("DeepSeek Local Chat")
        self.resize(900, 750)

        # DARK THEME
        self.setStyleSheet("""
        QWidget {
            background-color: #1a1a1a;
            color: white;
            font-family: Segoe UI;
        }

        QTextEdit {
            background-color: #222;
            border-radius: 8px;
            padding: 12px;
        }

        QPushButton {
            background-color: #333;
            border-radius: 8px;
            padding: 10px;
            font-size: 15px;
        }

        QPushButton:hover {
            background-color: #444;
        }
        """)

        self.messages = []
        self.remember_mode = True

        main_layout = QVBoxLayout()
        main_layout.setContentsMargins(10, 10, 10, 10)
        main_layout.setSpacing(10)

        # CHAT DISPLAY (expands to fill remaining space)
        self.chat_display = QTextEdit()
        self.chat_display.setReadOnly(True)
        self.chat_display.setFont(QFont("Segoe UI", 15))
        self.chat_display.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.chat_display.setStyleSheet("background-color:#1a1a1a;")
        main_layout.addWidget(self.chat_display)

        # BOTTOM PANEL (fixed height)
        bottom_panel = QWidget()
        bottom_panel.setFixedHeight(400)
        bottom_panel_layout = QHBoxLayout()
        bottom_panel_layout.setContentsMargins(0, 0, 0, 0)
        bottom_panel_layout.setSpacing(10)

        # PROMPT BOX
        self.input_box = QTextEdit()
        self.input_box.setFont(QFont("Segoe UI", 14))
        self.input_box.setPlaceholderText("Write your prompt here...")
        self.input_box.setStyleSheet("background-color:#222; border-radius:8px;")
        bottom_panel_layout.addWidget(self.input_box)

        # CONTROL PANEL (SEND + MEMORY)
        control_layout = QVBoxLayout()
        control_layout.setSpacing(10)

        # SEND BUTTON
        self.send_button = QPushButton("SEND")
        self.send_button.setMinimumHeight(60)
        self.send_button.setStyleSheet("""
        QPushButton{
            background-color:#4CAF50;
            font-size:18px;
            font-weight:bold;
            border-radius:10px;
        }
        QPushButton:hover{
            background-color:#45a049;
        }
        """)
        control_layout.addWidget(self.send_button)

        # MEMORY BUTTONS
        self.remember_button = QPushButton("Remember Chat")
        self.remember_button.setMinimumHeight(45)
        self.nomatter_button = QPushButton("No Memory")
        self.nomatter_button.setMinimumHeight(45)
        control_layout.addWidget(self.remember_button)
        control_layout.addWidget(self.nomatter_button)
        control_layout.addStretch()

        bottom_panel_layout.addLayout(control_layout)
        bottom_panel.setLayout(bottom_panel_layout)
        main_layout.addWidget(bottom_panel)

        self.setLayout(main_layout)

        # CONNECTIONS
        self.send_button.clicked.connect(self.send_message)
        self.remember_button.clicked.connect(self.enable_memory)
        self.nomatter_button.clicked.connect(self.disable_memory)

    def enable_memory(self):
        self.remember_mode = True
        self.chat_display.append("<font color='cyan'>Memory Mode Enabled</font><br>")

    def disable_memory(self):
        self.remember_mode = False
        self.chat_display.append("<font color='orange'>No Memory Mode Enabled</font><br>")

    def send_message(self):
        user_text = self.input_box.toPlainText().strip()
        if not user_text:
            return

        self.input_box.clear()
        self.chat_display.append(f"<b><font color='#00ff9c'>You:</font></b> {user_text}<br>")
        self.messages.append({"role": "user", "content": user_text})
        self.chat_display.append("<b><font color='#aaaaaa'>🤖 Thinking...</font></b>")
        self.chat_display.moveCursor(self.chat_display.textCursor().End)

        self.worker = AIWorker(self.messages, self.remember_mode)
        self.worker.token_received.connect(self.update_stream)
        self.worker.finished.connect(self.finish_response)
        self.current_response = ""
        self.worker.start()

    def update_stream(self, token):
        cursor = self.chat_display.textCursor()
        cursor.movePosition(cursor.End)
        cursor.insertText(token)
        self.chat_display.setTextCursor(cursor)
        self.chat_display.ensureCursorVisible()
        self.current_response += token

    def finish_response(self):
        self.chat_display.append("<br>")
        if self.remember_mode:
            self.messages.append({"role": "assistant", "content": self.current_response})


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = ChatApp()
    window.showMaximized()
    sys.exit(app.exec_())

SystemExit: 0

C:\Users\USER\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
